# Input-Length Sweep: MRINN vs MR-LSTM vs MR-GRU

**Self-contained.** Nothing to install beyond `holidays`; no repo to clone; runs on Kaggle, Colab or locally.

This notebook trains **three models** across **seven input-window lengths** and produces the two figures:

| | |
|---|---|
| **Chart A** | actual imbalance price vs. the median forecast of all three models |
| **Chart B** | AQL, AQCR, MAE, RMSE against `T`, one line per model |

### The one thing that differs between the three models

All three share an identical market-rule head, quantile head, loss, data pipeline and splits.
**Only the encoder changes:**

```
MRINN     x_f (N, T)     ──Dense(H) x2──▶  h_f (N, H)      params GROW with T
MR-LSTM   x_f (N, T, 1)  ──LSTM(H)  ─────▶  h_f (N, H)      params CONSTANT in T
MR-GRU    x_f (N, T, 1)  ──GRU(H)   ─────▶  h_f (N, H)      params CONSTANT in T
                                              │
                       ┌──────────────────────┴──────────────────────┐
                       │   IDENTICAL FOR ALL THREE                   │
                       │   get_P_RE · get_P_EX · get_P_SC            │
                       │   smooth_min/max + final_gate               │
                       │   HierarchicalQuantileHeadQ50 ──▶ (N, Q)    │
                       └─────────────────────────────────────────────┘
```

### Configuration

- `T ∈ {1, 4, 8, 16, 32, 64, 96}` — 15 min to 24 h of history
- 3 models x 7 windows = **21 runs**, 50 epochs, seed 42
- Test window 2025-09-01 → 2026-01-01 (11,697 intervals), unchanged from the reference results
- **Runtime ≈ 3.2 h.** Resumable — every finished run is written to disk immediately.

> **Run this on Kaggle with *Save & Run All*.** It executes headless, so a 3-hour sweep
> survives without a browser tab open. Free Colab disconnects after ~90 minutes idle.
> Use a **CPU** session: the models are 17 RNNs at 8 hidden units, far too small for a GPU
> to pay off, and the CPU session gives 30 GB RAM which `T=96` appreciates.

---
> **Base model:** Yu et al., *A Market-Rule-Informed Neural Network for Efficient Imbalance
> Electricity Price Forecasting*, Advanced Engineering Informatics 76 (2026) 105083 —
> [github.com/runyao-yu/MRINN](https://github.com/runyao-yu/MRINN)

## 1. Environment, paths and dependencies

Detects Kaggle / Colab / local and sets the data and results paths accordingly.

**Data.** On Kaggle, upload `imbalance_data.csv` (28 MB) once as a Dataset and attach it —
faster and more reliable than cloning MRINN, whose git pack is 157 MB. The cell searches
common locations and falls back to cloning, then to a manual upload prompt.

In [ ]:
import os, sys, gc, json, time, warnings, subprocess
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------- where are we ----------
ON_KAGGLE = Path("/kaggle/input").exists()
ON_COLAB  = "google.colab" in sys.modules

if ON_KAGGLE:   ENV = "Kaggle"
elif ON_COLAB:  ENV = "Colab"
else:           ENV = "Local"

# ---------- dependencies ----------
# Nothing to install: everything below ships with Kaggle and Colab images.
# The notebook therefore runs with Internet OFF, provided the dataset is attached.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from keras import ops as K
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score

# root_mean_squared_error is sklearn >= 1.4; fall back for older builds
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(a, b): return float(np.sqrt(mean_squared_error(a, b)))

# ---------- results directory ----------
if ON_KAGGLE:
    OUT = Path("/kaggle/working")
elif ON_COLAB:
    OUT = Path("/content/drive/MyDrive/mrlstm_sweep") if Path("/content/drive").exists() else Path("/content/mrlstm_sweep")
else:
    OUT = Path("./sweep_results")
OUT.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUT / "sweep_results.csv"
PRED_NPZ    = OUT / "sweep_predictions.npz"
CKPT_DIR    = OUT / "ckpt"; CKPT_DIR.mkdir(exist_ok=True)

# ---------- locate the dataset ----------
# Paste an explicit path here to skip the search entirely.
DATA_PATH_OVERRIDE = ""

def _find_dataset():
    if DATA_PATH_OVERRIDE and Path(DATA_PATH_OVERRIDE).exists():
        return Path(DATA_PATH_OVERRIDE)

    # exact known locations first
    for p in (Path("MRINN/Data/imbalance_data.csv"),
              Path("external/MRINN/Data/imbalance_data.csv"),
              Path("../mrinn/MRINN/Data/imbalance_data.csv"),
              Path("imbalance_data.csv")):
        if p.exists():
            return p

    # then search recursively - Kaggle nests dataset files at unpredictable depths,
    # and the upload may have been renamed
    roots = [Path("/kaggle/input"), Path("/content"), Path("./data")]
    csvs = [c for r in roots if r.exists() for c in r.rglob("*.csv")]
    named = [c for c in csvs if "imbalance" in c.name.lower()]
    if named:
        return named[0]
    if len(csvs) == 1:            # exactly one CSV attached - almost certainly it
        return csvs[0]
    return None


DATA_PATH = _find_dataset()

if DATA_PATH is None and not ON_KAGGLE:
    print("Dataset not found locally - cloning MRINN (157 MB, one-off)...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/runyao-yu/MRINN"], check=False)
    if Path("MRINN/Data/imbalance_data.csv").exists():
        DATA_PATH = Path("MRINN/Data/imbalance_data.csv")
    elif ON_COLAB:
        from google.colab import files
        files.upload()
        DATA_PATH = _find_dataset()

if DATA_PATH is None:
    ki = Path("/kaggle/input")
    listing = ("\n".join(f"    {p}" for p in sorted(ki.rglob("*"))[:40])
               if ki.exists() and any(ki.iterdir()) else "    (empty - no dataset attached)")
    raise FileNotFoundError(
        "imbalance_data.csv not found.\n\n"
        f"Contents of /kaggle/input:\n{listing}\n\n"
        "On Kaggle: right-hand panel -> Input -> '+ Add Input' -> select your uploaded\n"
        "dataset, then re-run this cell. If the file is there under another name, set\n"
        "DATA_PATH_OVERRIDE at the top of this cell to its full path."
    )

# keras.ops is a Keras 3 API (TensorFlow >= 2.16). Fail loudly and early rather
# than with a cryptic AttributeError forty minutes into a sweep.
_kv = tuple(int(x) for x in keras.__version__.split(".")[:2])
assert _kv >= (3, 0), (
    f"needs Keras 3 (TensorFlow >= 2.16); this image has Keras {keras.__version__}. "
    "On Kaggle pick a newer environment under Settings > Environment."
)

print(f"environment : {ENV}")
print(f"tensorflow  : {tf.__version__}   keras {keras.__version__}")
print(f"data        : {DATA_PATH}  ({DATA_PATH.stat().st_size/1e6:.1f} MB)")
print(f"results     : {OUT}")
if ENV == "Colab" and not Path("/content/drive").exists():
    print("\n  WARNING: Drive is not mounted. Results live in /content and are lost on disconnect.")
    print("           Run: from google.colab import drive; drive.mount('/content/drive')")

## 2. Reproducibility

One seed pins Python, NumPy and TensorFlow, and enables deterministic op ordering so the
same run reproduces exactly. **Every result below is a single seed (42)** — see the caveat
in §11 about what that does and does not license you to claim.

In [ ]:
import random

def set_random_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

SEED = 42
set_random_seed(SEED)
print(f"seed = {SEED}")

## 3. Data pipeline

Ported unchanged from MRINN's `library_imbalance/data.py`.

Splits are **chronological, never shuffled** — adjacent 15-minute intervals are near
duplicates, so random splitting would leak badly. Lag columns are built **within each
split separately**, so a test row's history never reaches back into validation data, and
the scalers are fit on **train only**.

> **Scaling note.** MR-LSTM's standalone notebook uses a shared-lag scaler (one scaler per
> signal instead of per column). This sweep deliberately uses MRINN's original
> **per-column** `scale_data` for *all three* models. Changing the encoder and the
> preprocessing at once would confound the comparison, and per-column keeps these numbers
> directly comparable to the reference results.

In [ ]:
FEATS_PRICES = ["P_aFRR_pos", "P_mFRR_pos", "P_aFRR_neg", "P_mFRR_neg",
                "P_aFRR_pos_MOL", "P_aFRR_neg_MOL",
                "P_ID15_nemo", "P_ID60_nemo", "P_DA_nemo"]
FEATS_CAPACITIES = ["L_ID15", "L_ID60", "L_DA"]
FEATS_VOLUME = ["system_imbalance", "E_aFRR_pos", "E_mFRR_pos", "E_aFRR_neg", "E_mFRR_neg"]
FEATS = FEATS_PRICES + FEATS_CAPACITIES + FEATS_VOLUME
LABEL = ["imbalance_price"]

TRAIN_RANGE = ("2022-01-01 00:00:00+00:00", "2025-05-01 00:00:00+00:00")
VAL_RANGE   = ("2025-05-01 00:00:00+00:00", "2025-09-01 00:00:00+00:00")
TEST_RANGE  = ("2025-09-01 00:00:00+00:00", "2026-01-01 00:00:00+00:00")

TIME_COL = "Time [UTC] start"


def load_data(path):
    df = pd.read_csv(path, parse_dates=[TIME_COL, "Time [UTC] end"])
    df.rename(columns={"P_VoAA_pos": "P_aFRR_pos_MOL",
                       "P_VoAA_neg": "P_aFRR_neg_MOL"}, inplace=True)
    df["L_DA"] = 0          # pinned to zero upstream; its encoder gets pruned
    return df


def split_data(df, train_range, val_range, test_range):
    def m(r):
        s, e = pd.to_datetime(r[0]), pd.to_datetime(r[1])
        return (df[TIME_COL] >= s) & (df[TIME_COL] < e)      # half-open
    return df.loc[m(train_range)].copy(), df.loc[m(val_range)].copy(), df.loc[m(test_range)].copy()


def make_shifts(df, cols, lags):
    # build every lag column at once: inserting one at a time fragments the frame,
    # which is painful at T=96 (17 x 96 = 1,632 columns)
    new = {f"{c}_lag{L}": df[c].shift(L) for c in cols if c in df.columns for L in lags}
    return pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)


def shift_data(df_train, df_val, df_test, target_col, feature_cols, lags):
    lag_cols = [f"{c}_lag{L}" for c in feature_cols for L in lags]
    needed = lag_cols + [target_col, TIME_COL]
    out = []
    for d in (df_train, df_val, df_test):
        s = make_shifts(d, feature_cols, lags)
        s = s.loc[:, ~s.columns.duplicated(keep="first")]
        out.append(s[needed].dropna().copy())
    return out[0], out[1], out[2], lag_cols


def scale_data(tr, va, te, feature_names, target_col):
    Xtr, Xva, Xte = (f[feature_names].to_numpy(float) for f in (tr, va, te))
    ytr, yva, yte = (f[target_col].to_numpy(float).reshape(-1, 1) for f in (tr, va, te))

    x_scaler = RobustScaler().fit(Xtr)          # fit on TRAIN only
    y_scaler = RobustScaler().fit(ytr)

    mk = lambda a: pd.DataFrame(x_scaler.transform(a), columns=feature_names)
    mky = lambda a: pd.DataFrame(y_scaler.transform(a).ravel(), columns=target_col)
    return mk(Xtr), mk(Xva), mk(Xte), mky(ytr), mky(yva), mky(yte), y_scaler


def scale_param(df_train, cols, param_dict):
    stacked = pd.concat([df_train[c] for c in cols], axis=0,
                        ignore_index=True).to_numpy().reshape(-1, 1)
    sc = RobustScaler().fit(stacked)
    return {k: float(sc.transform(np.array([[v]], float))[0, 0]) for k, v in param_dict.items()}


REGELZONEN = load_data(DATA_PATH)
DF_TRAIN, DF_VAL, DF_TEST = split_data(REGELZONEN, TRAIN_RANGE, VAL_RANGE, TEST_RANGE)

print(f"rows  train {len(DF_TRAIN):>7,}   val {len(DF_VAL):>6,}   test {len(DF_TEST):>6,}")
print(f"test window  {DF_TEST[TIME_COL].min()}  ->  {DF_TEST[TIME_COL].max()}")
print(f"test price   mean {DF_TEST[LABEL[0]].mean():7.2f}   std {DF_TEST[LABEL[0]].std():7.2f}"
      f"   max {DF_TEST[LABEL[0]].max():9.2f}")

## 4. Regulatory constants C0 – C10

These are the **real thresholds from the settlement rulebook** — a 50 MW dead band, the
200/800 MW scarcity band, a EUR 1000/MWh cap — not learned parameters. They are pushed
through the same `RobustScaler` statistics as the features they get compared against.

Freezing them is the point of the architecture: capacity a black-box model would spend
rediscovering these numbers is left free to estimate the latent market state.

In [ ]:
PARAM_PRICES     = {"C1": 5, "C2": 10, "C3": 15, "C10": 1000}
PARAM_CAPACITIES = {"C4": 50, "C5": 200, "C6": 200, "C7": 200, "C8": 800, "C9": 1000}

sp = scale_param(DF_TRAIN, FEATS_PRICES, PARAM_PRICES)
sc = scale_param(DF_TRAIN, FEATS_CAPACITIES, PARAM_CAPACITIES)

C = {"C0": 0.1, **sp, **sc}
C_ORDERED = [C[f"C{i}"] for i in range(11)]

print("scaled regulatory constants")
for i in range(11):
    raw = {**PARAM_PRICES, **PARAM_CAPACITIES}.get(f"C{i}", 0.1)
    print(f"  C{i:<2} raw {raw:>6}   scaled {C[f'C{i}']:+.4f}")

## 5. Differentiable market rules

Backpropagation needs the rulebook to be smooth, so each non-differentiable primitive is
replaced by a surrogate, and **every hard `if` becomes a softmax gate** over the branches it
chooses between — all branches are computed, then blended by learned weights conditioned on
the state the regulation conditions on.

| rule | what it is |
|---|---|
| `P_RE` | volume-weighted price of activated aFRR/mFRR reserves |
| `P_EX` | liquidity-weighted blend of day-ahead and intraday prices, with a directional markup |
| `P_SC` | cubic scarcity adder once the imbalance passes the onset threshold |
| final gate | system short → take the max; long → take the min |

**This entire section is identical for all three models.** It is imported verbatim from
MRINN and is the reason the sweep is a clean single-factor ablation.

In [ ]:
def smooth_abs(x, eps=1e-9):  return K.sqrt(K.square(x) + eps)
def smooth_sign(x):           return K.tanh(x)
def smooth_max(x, y):         return y + K.softplus(x - y)
def smooth_min(x, y):         return -smooth_max(-x, -y)

def gate_first(x):  return x[:, :1]
def gate_second(x): return x[:, 1:2]

def safe_div_pair(t, eps=1e-7):
    # NOTE: sign-unaware, exactly as upstream. A negative denominator yields a
    # wrong-signed fraction. Left as-is: matching MRINN matters more than fixing it,
    # and changing it would confound the comparison.
    P, L = t
    return P / smooth_max(K.abs(L), eps)

def get_weighted_frac_safe(numer, denom, label, hidden_units):
    return layers.Lambda(safe_div_pair, output_shape=(hidden_units,))([numer, denom])


# ---- A. balancing-energy price ------------------------------------------------
def get_P_RE(E_ap, E_mp, E_an, E_mn, V, P_ap, P_mp, P_an, P_mn, P_vp, P_vn, H):
    E_sum_pos = E_ap + E_mp
    E_sum_neg = E_an + E_mn

    P_act_pos = get_weighted_frac_safe(E_ap * P_ap + E_mp * P_mp, E_sum_pos, "RE_pos", H)
    P_act_neg = get_weighted_frac_safe(E_an * P_an + E_mn * P_mn, E_sum_neg, "RE_neg", H)

    gate_in = layers.Concatenate(name="gate_RE_in")(
        [layers.Activation("tanh")(E_sum_pos), layers.Activation("tanh")(E_sum_neg), V])
    W = layers.Dense(4, activation="softmax", name="gate_RE")(gate_in)

    onesH = K.ones_like(P_act_pos)
    return layers.Add(name="P_RE_rep")([
        layers.Multiply()([P_act_pos, W[:, 0:1] * onesH]),
        layers.Multiply()([P_act_neg, W[:, 1:2] * onesH]),
        layers.Multiply()([P_vp,      W[:, 2:3] * onesH]),
        layers.Multiply()([P_vn,      W[:, 3:4] * onesH]),
    ])


# ---- B. market-reference price -------------------------------------------------
def ramp_function(P_rep, V_rep, C4, label):
    C4_t = K.ones_like(V_rep) * K.cast(C4, V_rep.dtype)
    gate_in = layers.Concatenate(name=f"gate_ramp_in_{label}")([V_rep, C4_t])
    W = layers.Dense(3, activation="softmax", name=f"gate_ramp_{label}")(gate_in)
    onesH = K.ones_like(P_rep)
    return (-onesH * (W[:, 0:1] * onesH)
            + (V_rep / (C4_t + 1e-7)) * onesH * (W[:, 1:2] * onesH)
            + onesH * (W[:, 2:3] * onesH))


def get_P_EX(P15, L15, P60, L60, PDA, V, C0, C1, C2, C3, C4, C5, C6):
    r15, r60, rDA = (ramp_function(p, V, C4, n)
                     for p, n in ((P15, "ID15"), (P60, "ID60"), (PDA, "DA")))

    C0_t = K.cast(C0, P15.dtype)
    m1 = smooth_max(K.ones_like(P15) * K.cast(C1, P15.dtype), C0_t * smooth_abs(P15))
    m2 = smooth_max(K.ones_like(P60) * K.cast(C2, P60.dtype), C0_t * smooth_abs(P60))
    m3 = smooth_max(K.ones_like(PDA) * K.cast(C3, PDA.dtype), C0_t * smooth_abs(PDA))

    P15_mkt, P60_mkt, PDA_mkt = P15 + r15 * m1, P60 + r60 * m2, PDA + rDA * m3

    one = K.ones_like(L15)
    w15 = smooth_min(one, L15 / K.cast(C5, L15.dtype))
    w60 = smooth_min(one - w15, L60 / K.cast(C6, L60.dtype))
    wDA = one - w15 - w60

    P_EX = (P15_mkt * (w15 * K.ones_like(P15_mkt))
            + P60_mkt * (w60 * K.ones_like(P60_mkt))
            + PDA_mkt * (wDA * K.ones_like(PDA_mkt)))

    P_base = layers.Add(name="P_EX_basis")([
        layers.Multiply()([P15, w15]),
        layers.Multiply()([P60, w60]),
        layers.Multiply()([PDA, wDA]),
    ])
    return P_EX, P_base


# ---- C. scarcity function ------------------------------------------------------
def get_P_SC(P_base, V, C7, C8, C9, C10):
    aV, sV = smooth_abs(V), smooth_sign(V)
    t = lambda c: K.ones_like(aV) * K.cast(c, aV.dtype)
    C7_t, C8_t, C9_t, C10_t = t(C7), t(C8), t(C9), t(C10)
    denom = (C9_t - C7_t) + 1e-7

    term0 = P_base
    term1 = P_base + sV * C10_t * K.power((aV - C7_t) / denom, 3.0)
    term2 = P_base + sV * C10_t * K.power((C8_t - C7_t) / denom, 3.0)

    W = layers.Dense(3, activation="softmax", name="Psc_gate")(
        layers.Concatenate(name="Psc_gate_in")([aV, C7_t, C8_t]))

    return layers.Add(name="Psc_mix")([
        layers.Multiply()([term0, W[:, 0:1]]),
        layers.Multiply()([term1, W[:, 1:2]]),
        layers.Multiply()([term2, W[:, 2:3]]),
    ])

print("market-rule surrogates defined (identical for all three models)")

## 6. The three models

One builder, one branch. `_encode()` is the **entire** difference between MRINN and the
recurrent variants — everything after it is shared code operating on identically-shaped
latent vectors.

Note the parameter arithmetic, which is the whole argument:

- **Dense encoder:** first layer is `T x H + H` weights **per signal**, so the model grows
  linearly with the window — 1,799 params at `T=1` up to roughly 14,000 at `T=96`.
- **Recurrent encoder:** the same cell is re-applied at every timestep, so the count is
  `constant in T` — 5,511 (LSTM) and 4,615 (GRU) at every window length.

**Only 16 of the 17 encoders carry parameters.** `get_P_EX` never consumes `L_DA`'s
representation (it derives `w_DA = 1 - w_ID15 - w_ID60`), so Keras prunes that branch, and
`load_data` pins the column to zero anyway. The input still exists and must still be fed.
Inherited from MRINN, left as-is.

In [ ]:
# (dataframe column, model input name) - ORDER IS POSITIONAL.
# The rule blocks read these tensors by position; a permutation still trains, it just
# feeds liquidity into a price slot and silently produces a wrong model.
FEATURE_SPEC = [
    ("system_imbalance", "system_imbalance"),
    ("E_aFRR_pos", "E_aFRR_pos_in"), ("E_mFRR_pos", "E_mFRR_pos_in"),
    ("P_aFRR_pos", "P_aFRR_pos_in"), ("P_mFRR_pos", "P_mFRR_pos_in"),
    ("E_aFRR_neg", "E_aFRR_neg_in"), ("E_mFRR_neg", "E_mFRR_neg_in"),
    ("P_aFRR_neg", "P_aFRR_neg_in"), ("P_mFRR_neg", "P_mFRR_neg_in"),
    ("P_aFRR_pos_MOL", "P_VoAA_pos_in"), ("P_aFRR_neg_MOL", "P_VoAA_neg_in"),
    ("P_ID15_nemo", "P_ID15_nemo_in"), ("P_ID60_nemo", "P_ID60_nemo_in"),
    ("P_DA_nemo", "P_DA_nemo_in"),
    ("L_ID15", "L_ID15_in"), ("L_ID60", "L_ID60_in"), ("L_DA", "L_DA_in"),
]
FEATURE_BASES = [b for b, _ in FEATURE_SPEC]
INPUT_NAMES   = [n for _, n in FEATURE_SPEC]
QUANTILES     = [0.1, 0.25, 0.5, 0.75, 0.9]


def make_inputs(frame, lags, kind):
    # MRINN wants flat (N, T); the recurrent models want (N, T, 1) oldest-step-first.
    arrs = []
    for base in FEATURE_BASES:
        a = frame[[f"{base}_lag{L}" for L in lags]].to_numpy("float32")   # newest -> oldest
        if kind != "mrinn":
            a = np.ascontiguousarray(a[:, ::-1])[:, :, None]              # oldest -> newest
        arrs.append(a)
    return arrs


def assert_input_alignment(model, arrays, expect_ndim):
    names = [t.name.split(":")[0] for t in model.inputs]
    if names != INPUT_NAMES:
        bad = [(i, a, b) for i, (a, b) in enumerate(zip(names, INPUT_NAMES)) if a != b]
        raise ValueError(f"input name order mismatch at {bad}")
    for n, a in zip(names, arrays):
        if a.ndim != expect_ndim:
            raise ValueError(f"input '{n}' has shape {a.shape}, expected ndim {expect_ndim}")


def multi_quantile_pinball_loss(quantiles):
    qs = tf.reshape(tf.constant(quantiles, tf.float32), (1, -1))
    def loss(y_true, y_pred):
        y_true = tf.reshape(tf.cast(y_true, tf.float32), (-1, 1))
        e = y_true - y_pred
        return tf.reduce_mean(tf.maximum(qs * e, (qs - 1.0) * e))
    return loss


def HierarchicalQuantileHeadQ50(z, quantiles, prefix="hq"):
    # Predict the median, then step outward with softplus-constrained (strictly
    # non-negative) residuals. Quantile crossing becomes arithmetically unreachable
    # rather than something training has to learn.
    sq = sorted(quantiles)
    if 0.5 not in sq:
        raise ValueError("requires quantile 0.5")
    mid = sq.index(0.5)
    nm = lambda q: f"{int(round(q * 100)):02d}"

    med = layers.Dense(1, activation="linear", name=f"{prefix}_q{nm(0.5)}")(z)
    out = {0.5: med}

    prev = med
    for q in sq[mid + 1:]:
        r = layers.Activation("softplus", name=f"{prefix}_pos_r{nm(q)}")(
            layers.Dense(1, name=f"{prefix}_r{nm(q)}")(z))
        prev = layers.Add(name=f"{prefix}_q{nm(q)}")([prev, r]); out[q] = prev

    prev = med
    for q in reversed(sq[:mid]):
        r = layers.Activation("softplus", name=f"{prefix}_pos_r{nm(q)}")(
            layers.Dense(1, name=f"{prefix}_r{nm(q)}")(z))
        prev = layers.Subtract(name=f"{prefix}_q{nm(q)}")([prev, r]); out[q] = prev

    return layers.Concatenate(axis=-1, name=f"{prefix}_output")([out[q] for q in quantiles])


# ================== THE ONLY PART THAT DIFFERS ==================
def _encode(x, kind, H, num_layer, name):
    if kind == "mrinn":
        for i in range(num_layer):
            x = layers.Dense(H, activation="swish", name=f"{name}_dense_{i}")(x)
        return x
    Cell = layers.LSTM if kind == "lstm" else layers.GRU
    for i in range(num_layer):
        x = Cell(H, return_sequences=(i < num_layer - 1), name=f"{name}_{kind}_{i}")(x)
    return x
# ================================================================


def build_model(kind, T, H=8, num_layer=None, quantiles=QUANTILES, lr=1e-3):
    kind = kind.lower()
    assert kind in {"mrinn", "lstm", "gru"}
    # MRINN's reference config stacks two Dense layers; the recurrent models use one,
    # because two stacked recurrent layers costs ~15k params and blows the budget.
    if num_layer is None:
        num_layer = 2 if kind == "mrinn" else 1

    shape = (T,) if kind == "mrinn" else (T, 1)
    feats_in = [layers.Input(shape=shape, name=n) for n in INPUT_NAMES]
    reps = [_encode(x, kind, H, num_layer, base)
            for x, base in zip(feats_in, FEATURE_BASES)]

    (V, E_ap, E_mp, P_ap, P_mp, E_an, E_mn, P_an, P_mn,
     P_vp, P_vn, P15, P60, PDA, L15, L60, LDA) = reps

    # ---- everything below is identical for all three models ----
    P_RE = get_P_RE(E_ap, E_mp, E_an, E_mn, V, P_ap, P_mp, P_an, P_mn, P_vp, P_vn, H)
    P_EX, P_base = get_P_EX(P15, L15, P60, L60, PDA, V, *C_ORDERED[:7])
    P_SC = get_P_SC(P_base, V, *C_ORDERED[7:11])

    lo = smooth_min(smooth_min(P_RE, P_EX), P_SC)
    hi = smooth_max(smooth_max(P_RE, P_EX), P_SC)
    Wf = layers.Dense(2, activation="softmax", name="final_gate")(V)
    z = layers.Add(name="imbalance_price_rep")([
        layers.Multiply()([lo, Wf[:, 0:1]]),
        layers.Multiply()([hi, Wf[:, 1:2]]),
    ])

    out = HierarchicalQuantileHeadQ50(z, quantiles, prefix="imbalance_price_hq")

    model = models.Model(feats_in, out, name=f"{kind.upper()}_T{T}")
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss=multi_quantile_pinball_loss(quantiles))
    return model


# quick structural check
for k in ("mrinn", "lstm", "gru"):
    m = build_model(k, T=8)
    print(f"  {k:<6} T=8  params {m.count_params():>6,}   input {m.inputs[0].shape}")
    del m
tf.keras.backend.clear_session(); gc.collect()

## 7. Evaluation metrics

Ported from `library_imbalance/evaluation.py`. Everything is computed **after** inverse
scaling, so the numbers are in EUR/MWh.

| metric | what it measures |
|---|---|
| **AQL** | mean pinball loss over the five quantiles — overall quality of the *distribution* |
| **AQCR** | quantile crossing rate — % of predictions where a lower quantile exceeds a higher one |
| **AQCE** | coverage error — does the "80% band" actually contain the truth 80% of the time |
| **AIW** | mean interval width — a band only counts if it did not get wider to buy coverage |
| **MAE / RMSE / R²** | point accuracy, computed on the median prediction |

In [ ]:
def pinball_loss_np(y, p, q):
    e = np.asarray(y).ravel() - np.asarray(p).ravel()
    return float(np.mean(np.maximum(q * e, (q - 1.0) * e)))


def AQCR_percent(P):
    P = np.asarray(P)
    if P.shape[-1] < 2:
        return 0.0
    return float((P[..., :-1] > P[..., 1:]).mean() * 100.0)


def AQCE_percent(y, P, quantiles):
    qs = np.asarray(quantiles, float); order = np.argsort(qs)
    qs, P = qs[order], np.asarray(P)[..., order]
    errs = []
    for i in range(len(qs) // 2):
        lo, hi = P[..., i], P[..., -(i + 1)]
        cov = float(((y >= lo) & (y <= hi)).mean())
        errs.append(abs(cov - float(qs[-(i + 1)] - qs[i])))
    return float(np.mean(errs) * 100.0) if errs else 0.0


def AIW_metric(P, quantiles):
    qs = np.asarray(quantiles, float); order = np.argsort(qs)
    qs, P = qs[order], np.asarray(P)[..., order]
    w = [np.mean(P[..., -(i + 1)] - P[..., i]) for i in range(len(qs) // 2)]
    return float(np.mean(w)) if w else 0.0


def evaluate_performance(y_true_scaled, yqs_scaled, quantiles, y_scaler):
    inv = lambda a: y_scaler.inverse_transform(np.asarray(a).reshape(-1, 1)).ravel()
    y = inv(np.asarray(y_true_scaled).ravel())
    yq = [inv(a) for a in yqs_scaled]
    P = np.column_stack(yq)

    mid = quantiles.index(0.5)
    r = {"RMSE": float(root_mean_squared_error(y, yq[mid])),
         "MAE":  float(mean_absolute_error(y, yq[mid])),
         "R2":   float(r2_score(y, yq[mid]))}
    r["AQL"]  = float(np.mean([pinball_loss_np(y, p, q) for q, p in zip(quantiles, yq)]))
    r["AQCR"] = AQCR_percent(P)
    r["AQCE"] = AQCE_percent(y, P, quantiles)
    r["AIW"]  = AIW_metric(P, quantiles)
    return r, y, yq[mid]

print("metrics defined")

## 8. Sweep driver

21 runs: 3 models x 7 window lengths, 50 epochs each, seed 42.

**Resumable.** Every finished run is appended to `sweep_results.csv` immediately and its
median test predictions are written to `sweep_predictions.npz`. Re-running the cell skips
anything already recorded, so a crash at run 18 does not cost the first 17.

**Data is prepared once per `T`, not once per run** — at `T=96` the lag construction builds
17 x 96 = 1,632 columns, and doing that three times per window would dominate the runtime.

> Set `SMOKE_TEST = True` first. It runs `T ∈ {1, 4}` for 2 epochs (~2 min) and exercises
> the whole path — data, both encoders, checkpointing, evaluation, both charts — before
> you commit three hours.

In [ ]:
# ----------------------------- configuration -----------------------------
SMOKE_TEST = False           # True -> T in {1,4}, 2 epochs, ~2 minutes

SWEEP_T    = [1, 4, 8, 16, 32, 64, 96]
MODELS     = ["mrinn", "lstm", "gru"]
EPOCHS     = 50
HIDDEN     = 8
BATCH      = 1024

if SMOKE_TEST:
    SWEEP_T, EPOCHS = [1, 4], 2
    RESULTS_CSV = OUT / "smoke_results.csv"
    PRED_NPZ    = OUT / "smoke_predictions.npz"

LABELS = {"mrinn": "MRINN", "lstm": "MR-LSTM", "gru": "MR-GRU"}
# -------------------------------------------------------------------------


def load_results():
    return pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()

def already_done(runs, kind, T):
    return (not runs.empty) and bool(((runs.model == kind) & (runs["T"] == T)).any())

def append_run(row):
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a",
                               header=not RESULTS_CSV.exists(), index=False)

PRED = dict(np.load(PRED_NPZ, allow_pickle=False)) if PRED_NPZ.exists() else {}


def prepare(T):
    lags = list(range(1, T + 1))
    tr, va, te, names = shift_data(DF_TRAIN, DF_VAL, DF_TEST, LABEL[0], FEATS, lags)
    Xtr, Xva, Xte, ytr, yva, yte, y_scaler = scale_data(tr, va, te, names, LABEL)
    return dict(lags=lags, Xtr=Xtr, Xva=Xva, Xte=Xte, ytr=ytr, yva=yva, yte=yte,
                y_scaler=y_scaler, stamps=te[TIME_COL].to_numpy())


def run_one(kind, T, D):
    set_random_seed(SEED)
    tr_in = make_inputs(D["Xtr"], D["lags"], kind)
    va_in = make_inputs(D["Xva"], D["lags"], kind)
    te_in = make_inputs(D["Xte"], D["lags"], kind)

    model = build_model(kind, T, H=HIDDEN)
    assert_input_alignment(model, tr_in, expect_ndim=2 if kind == "mrinn" else 3)

    ckpt = str(CKPT_DIR / f"{kind}_T{T}.weights.h5")
    cb = tf.keras.callbacks.ModelCheckpoint(ckpt, monitor="val_loss", mode="min",
                                            save_best_only=True, save_weights_only=True)
    t0 = time.perf_counter()
    model.fit(tr_in, D["ytr"].to_numpy("float32"),
              validation_data=(va_in, D["yva"].to_numpy("float32")),
              epochs=EPOCHS, batch_size=BATCH, callbacks=[cb], verbose=0)
    train_sec = time.perf_counter() - t0

    model.load_weights(ckpt)                       # best epoch, not last
    yp = model.predict(te_in, verbose=0, batch_size=BATCH)
    yqs = [yp[:, j] for j in range(len(QUANTILES))]

    res, y_orig, y50 = evaluate_performance(D["yte"], yqs, QUANTILES, D["y_scaler"])
    row = {"model": kind, "T": T, **{k: res[k] for k in
           ["AQL", "AQCR", "AQCE", "AIW", "MAE", "RMSE", "R2"]},
           "Params": int(model.count_params()), "TrainSec": round(train_sec, 1),
           "n_test": len(y_orig)}

    PRED[f"{kind}_T{T}_q50"] = y50.astype("float32")
    PRED[f"{kind}_T{T}_q10"] = D["y_scaler"].inverse_transform(
        yqs[0].reshape(-1, 1)).ravel().astype("float32")
    PRED[f"{kind}_T{T}_q90"] = D["y_scaler"].inverse_transform(
        yqs[-1].reshape(-1, 1)).ravel().astype("float32")
    PRED[f"actual_T{T}"] = y_orig.astype("float32")
    PRED[f"stamps_T{T}"] = D["stamps"].astype("datetime64[ns]").astype("int64")

    del model, tr_in, va_in, te_in, yp, yqs
    tf.keras.backend.clear_session(); gc.collect()
    return row


# ------------------------------- run -------------------------------------
total = len(SWEEP_T) * len(MODELS)
done_n, t_start = 0, time.perf_counter()
print(f"sweep: {total} runs  |  T={SWEEP_T}  |  epochs={EPOCHS}  |  results -> {RESULTS_CSV}\n")

for T in SWEEP_T:
    runs = load_results()
    todo = [k for k in MODELS if not already_done(runs, k, T)]
    if not todo:
        print(f"T={T:<3} all models already done - skipping")
        done_n += len(MODELS)
        continue

    print(f"T={T:<3} preparing data ({17 * T} lag columns)...", flush=True)
    D = prepare(T)
    print(f"      rows train {len(D['Xtr']):,}  val {len(D['Xva']):,}  test {len(D['Xte']):,}")

    for kind in MODELS:
        runs = load_results()
        if already_done(runs, kind, T):
            done_n += 1
            continue
        row = run_one(kind, T, D)
        append_run(row)
        np.savez_compressed(PRED_NPZ, **PRED)
        done_n += 1

        el = time.perf_counter() - t_start
        eta = el / max(done_n, 1) * (total - done_n)
        print(f"      {LABELS[kind]:<8} AQL {row['AQL']:7.3f}  MAE {row['MAE']:6.2f}  "
              f"RMSE {row['RMSE']:6.2f}  AQCR {row['AQCR']:.2f}  "
              f"params {row['Params']:>6,}  {row['TrainSec']:>6.0f}s   "
              f"[{done_n}/{total}, eta {eta/60:.0f} min]", flush=True)

    del D; gc.collect()

print(f"\nsweep complete in {(time.perf_counter() - t_start)/60:.1f} min")
RUNS = load_results().sort_values(["T", "model"]).reset_index(drop=True)
RUNS

### Sanity checks

Four things that must hold. If any fails, the numbers below are not trustworthy.

In [ ]:
R = load_results()
ok = True

# 1. recurrent parameter counts are CONSTANT in T - the headline claim
for k in ("lstm", "gru"):
    n = R[R.model == k].Params.unique()
    good = len(n) == 1
    ok &= good
    print(f"[{'PASS' if good else 'FAIL'}] {LABELS[k]} params constant in T -> {sorted(n)}")

# 2. MRINN's parameters grow with T
mr = R[R.model == "mrinn"].sort_values("T")
if len(mr) > 1:
    good = bool((mr.Params.diff().dropna() > 0).all())
    ok &= good
    print(f"[{'PASS' if good else 'FAIL'}] MRINN params grow with T -> "
          f"{mr.Params.min():,} .. {mr.Params.max():,}")

# 3. zero quantile crossing everywhere - structural, not learned
good = bool((R.AQCR == 0).all()); ok &= good
print(f"[{'PASS' if good else 'FAIL'}] AQCR == 0 for all {len(R)} runs "
      f"(max {R.AQCR.max():.4f})")

# 4. pipeline control: at T=1 a recurrent encoder has no sequence to read, so it
#    should collapse to roughly the Dense encoder. A large gap means a bug, not a finding.
t1 = R[R["T"] == 1].set_index("model").AQL
if len(t1) == 3:
    spread = t1.max() - t1.min()
    if SMOKE_TEST:
        # 2 epochs converges nothing, so a wide spread here means nothing either
        print(f"[SKIP] T=1 control - not meaningful at {EPOCHS} epochs "
              f"(spread {spread:.3f} AQL)")
    else:
        good = spread < 0.5; ok &= good
        print(f"[{'PASS' if good else 'FAIL'}] T=1 control: all three within "
              f"{spread:.3f} AQL ({dict(t1.round(3))})")

print("\n" + ("all checks passed" if ok else "SOME CHECKS FAILED - investigate before using these results"))

## 9. Chart A — imbalance price: actual vs. all three models

Actual price against the **median forecast** of each model, over one week of the test set.

All three are plotted at the **same `T`**, so they have seen identical context and the
comparison is fair. Change `CHART_T` to any window in the sweep and `WEEK` to slide along
the test set — no retraining needed, the predictions are cached in the `.npz`.

In [ ]:
COL = {"actual": "#0b0b0b", "mrinn": "#a5762c", "lstm": "#2a78d6", "gru": "#eb6834"}

CHART_T   = max(SWEEP_T)     # all three models at the same context length
WEEK      = 4                # which week of the test set (0-based)
N_POINTS  = 672              # 672 x 15 min = 7 days
SHOW_BAND = None             # e.g. "gru" to overlay that model's 80% interval

P = dict(np.load(PRED_NPZ, allow_pickle=False))
stamps = pd.to_datetime(P[f"stamps_T{CHART_T}"])
actual = P[f"actual_T{CHART_T}"]

s = min(WEEK * N_POINTS, max(len(actual) - N_POINTS, 0))
sl = slice(s, s + N_POINTS)
x = stamps[sl]

fig, ax = plt.subplots(figsize=(15, 5.2))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

if SHOW_BAND:
    ax.fill_between(x, P[f"{SHOW_BAND}_T{CHART_T}_q10"][sl], P[f"{SHOW_BAND}_T{CHART_T}_q90"][sl],
                    color=COL[SHOW_BAND], alpha=0.13, linewidth=0, zorder=1,
                    label=f"{LABELS[SHOW_BAND]} 80% interval")

for k in MODELS:
    key = f"{k}_T{CHART_T}_q50"
    if key in P:
        ax.plot(x, P[key][sl], color=COL[k], linewidth=1.5, alpha=0.9,
                zorder=3, label=f"{LABELS[k]} (median)")

ax.plot(x, actual[sl], color=COL["actual"], linewidth=1.5, zorder=5, label="Actual")

ax.set_title(f"Imbalance price: actual vs. three market-rule models   (T = {CHART_T})",
             fontsize=13, fontweight="bold", loc="left", pad=14)
ax.text(0, 1.02, f"one week of the test set - {N_POINTS} x 15-minute intervals from "
                 f"{x.min():%Y-%m-%d}", transform=ax.transAxes, fontsize=9, color="#6f7d84")
ax.set_ylabel("EUR / MWh"); ax.set_xlabel("")
ax.grid(True, color="#e1e0d9", linewidth=0.6); ax.set_axisbelow(True)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.tick_params(length=0)
ax.legend(frameon=False, ncol=5, fontsize=9, loc="upper left", bbox_to_anchor=(0, -0.10))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(OUT / f"chart_price_T{CHART_T}.png", dpi=180, bbox_inches="tight",
            facecolor="white")
plt.show()

err = {LABELS[k]: float(np.mean(np.abs(P[f"{k}_T{CHART_T}_q50"][sl] - actual[sl])))
       for k in MODELS if f"{k}_T{CHART_T}_q50" in P}
print("MAE over the plotted week:  " + "   ".join(f"{k} {v:.2f}" for k, v in err.items()))
print("(one week is a small sample - judge the models on the full-test metrics in Chart B)")

## 10. Chart B — metrics against input window length

Each panel is one metric; each line is one model; the x-axis is how much history the model
was given. **This is the figure that carries the argument.**

Read the AQL and RMSE panels together: MRINN should climb (worse) as `T` grows while the
recurrent models fall or stay flat — and they do it at a *constant* parameter count.

> **AQCR will be a flat line at zero for all three models.** That is the correct result,
> not a broken panel: all three use the hierarchical quantile head, which makes crossing
> arithmetically impossible. It is shown because "still exactly zero at every window
> length" is worth demonstrating rather than asserting.

In [ ]:
R = load_results().sort_values(["model", "T"])

PANELS = [("AQL",  "Average Quantile Loss",   "quality of the whole distribution - lower is better"),
          ("AQCR", "Quantile Crossing Rate",  "% - structurally zero by construction"),
          ("MAE",  "Mean Absolute Error",     "EUR/MWh on the median - lower is better"),
          ("RMSE", "Root Mean Squared Error", "EUR/MWh - penalises spikes - lower is better")]

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.4))
fig.patch.set_facecolor("white")

for ax, (metric, title, sub) in zip(axes.ravel(), PANELS):
    ax.set_facecolor("white")
    for k in MODELS:
        d = R[R.model == k].sort_values("T")
        if d.empty:
            continue
        ax.plot(d["T"], d[metric], marker="o", markersize=5.5, linewidth=2,
                color=COL[k], label=LABELS[k],
                markerfacecolor="white", markeredgewidth=1.8)

    ax.set_xscale("log", base=2)
    ax.set_xticks(SWEEP_T); ax.set_xticklabels([str(t) for t in SWEEP_T])
    ax.minorticks_off()
    ax.set_title(title, fontsize=11.5, fontweight="bold", loc="left", pad=18)
    ax.text(0, 1.03, sub, transform=ax.transAxes, fontsize=8.5, color="#6f7d84")
    ax.set_xlabel("input window T  (15-min steps)", fontsize=9, color="#52514e")
    ax.grid(True, color="#e1e0d9", linewidth=0.6); ax.set_axisbelow(True)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)

    if metric == "AQCR":                       # otherwise matplotlib autoscales pure noise
        ax.set_ylim(-0.5, 1.0)
        ax.text(0.5, 0.5, "0.00 everywhere\nno crossing at any window length",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=10, color="#898781")

axes[0, 0].legend(frameon=False, fontsize=9.5, loc="upper left")
plt.tight_layout(h_pad=3.4, w_pad=2.6)
plt.savefig(OUT / "chart_metrics_vs_T.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# parameter cost of the window - the other half of the argument
fig, ax = plt.subplots(figsize=(7.2, 4.2))
fig.patch.set_facecolor("white"); ax.set_facecolor("white")

for k in MODELS:
    d = R[R.model == k].sort_values("T")
    if d.empty: continue
    ax.plot(d["T"], d.Params, marker="o", markersize=5.5, linewidth=2, color=COL[k],
            label=LABELS[k], markerfacecolor="white", markeredgewidth=1.8)

ax.axhline(8900, color="#898781", linestyle="--", linewidth=1.2)
ax.text(SWEEP_T[0], 9200, "MLP baseline (8,900) - the bar to stay under",
        fontsize=8.5, color="#6f7d84")
ax.set_xscale("log", base=2); ax.set_xticks(SWEEP_T)
ax.set_xticklabels([str(t) for t in SWEEP_T]); ax.minorticks_off()
ax.set_title("Parameter count vs. input window", fontsize=11.5, fontweight="bold",
             loc="left", pad=18)
ax.text(0, 1.03, "recurrent encoders re-read one set of weights, so cost is flat in T",
        transform=ax.transAxes, fontsize=8.5, color="#6f7d84")
ax.set_xlabel("input window T"); ax.set_ylabel("trainable parameters")
ax.grid(True, color="#e1e0d9", linewidth=0.6); ax.set_axisbelow(True)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.tick_params(length=0); ax.legend(frameon=False, fontsize=9.5)
plt.tight_layout()
plt.savefig(OUT / "chart_params_vs_T.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.show()

## 11. Results table and summary

In [ ]:
R = load_results()
piv = R.pivot(index="T", columns="model", values=["AQL", "MAE", "RMSE", "Params"])
piv = piv.reindex(columns=["mrinn", "lstm", "gru"], level=1)
print("AQL / MAE / RMSE / Params by window length\n")
print(piv.round(3).to_string())

print("\n\nbest configuration per model")
print("-" * 62)
for k in MODELS:
    d = R[R.model == k]
    if d.empty: continue
    b = d.loc[d.AQL.idxmin()]
    print(f"  {LABELS[k]:<8} T={int(b['T']):<3} AQL {b.AQL:7.3f}   MAE {b.MAE:6.2f}   "
          f"RMSE {b.RMSE:6.2f}   R2 {b.R2:.3f}   params {int(b.Params):>6,}")

mr = R[R.model == "mrinn"].sort_values("T")
if len(mr) > 1:
    d0, d1 = mr.iloc[0], mr.iloc[-1]
    print(f"\nMRINN from T={int(d0['T'])} to T={int(d1['T'])}: "
          f"AQL {d0.AQL:.3f} -> {d1.AQL:.3f} ({100*(d1.AQL-d0.AQL)/d0.AQL:+.1f}%), "
          f"params {int(d0.Params):,} -> {int(d1.Params):,}")
for k in ("lstm", "gru"):
    d = R[R.model == k].sort_values("T")
    if len(d) > 1:
        d0, d1 = d.iloc[0], d.iloc[-1]
        print(f"{LABELS[k]} from T={int(d0['T'])} to T={int(d1['T'])}: "
              f"AQL {d0.AQL:.3f} -> {d1.AQL:.3f} ({100*(d1.AQL-d0.AQL)/d0.AQL:+.1f}%), "
              f"params constant at {int(d1.Params):,}")

print(f"\nartifacts written to {OUT}:")
for f in sorted(OUT.glob("*.png")) + sorted(OUT.glob("*.csv")) + sorted(OUT.glob("*.npz")):
    print(f"  {f.name:<34} {f.stat().st_size/1024:>8.0f} KB")

### How to read these results honestly

**This is one seed.** Reference multi-seed runs at `T=8` show a spread of ±0.11 to ±0.26
AQL across seeds 42–46. **A difference smaller than roughly 0.2 AQL at any single `T` is
inside the noise** and should not be reported as a win. What the sweep supports is the
*trend across window lengths*, which is far more robust than any individual point — and
that trend is the finding.

**Three further caveats worth stating alongside any number above:**

1. **The test window is calm.** Price std ≈ 83 against ≈ 387 over the full record, with
   only a handful of intervals above EUR 1000/MWh. The scarcity rule `P_SC` — the component
   most likely to separate these models — is barely exercised. Absolute errors here are
   low for that reason, and **must not be compared to MRINN's published table**, which was
   computed on a different split.
2. **The grid is coarse between T=8 and T=32**, which is exactly where the curve bends most
   sharply. Add intermediate points if that region matters to your argument.
3. **Recurrent models train roughly 3x slower**, and the gap widens with `T` because
   recurrence is sequential. Immaterial for a model retrained daily; material for one
   retrained per interval.

**And one thing the sweep does not change:** every model here forecasts **one step ahead**
(15 minutes). `T` is how far *back* the model looks, never how far *forward* it predicts.
A day-ahead forecast needs a shifted target and a different treatment of the market rules,
since tomorrow's settlement inputs do not exist at forecast time.